# Systems that are not DAGs

A causal DAG is a *solved* model: it assumes you can order the variables so every arrow
points forward. Two ordinary situations break that assumption without being ill-posed —
**simultaneity** (two quantities determined together) and **time structure** (yesterday's
value entering today's equation). `axiom.dynamics` takes such a system as a declarative
`Spec` and *compiles* it into ordinary `axiom.core.expr` trees, so there is still exactly
one `forward()`.

In [ ]:
from axiom.core import D, Param, dimensionless, latex, value
from axiom.dynamics import (
    AffineForm, Block, BlockOrder, BlockSolution, DynamicEquation, DynamicSystem, DynamicsError,
    Form, LeadNotSupportedError, ParseError, Role, SolveMethod, Unrolled, Variable, affine_split,
    block_order, conditional_form, lag_ref, lagged_columns, parse_equations, parse_ref,
    parse_system, prepare_panel, reduced_form, refs_in, solve_block, strongly_connected_components,
    time_ref, to_model_spec, unroll, unrolled_edges,
)
import numpy as np
import pandas as pd

from axiom.display import enable, show_math

enable();  # every axiom result renders itself from here on

NONE = dimensionless()

## Declaring a system

Variables carry a dimension and a role: `endogenous` means an equation determines it,
`exogenous` means it is supplied. `observed=False` marks a latent state. `initial` is the
value lags take before the first period.

The equations get a small syntax — that is the part that is unreadable as a tree. Names
resolve to a parameter if one is declared and to a variable otherwise; `stock[t-1]` (or
`stock.l1`) is a lag.

In [ ]:
system = parse_system(
    "stock = decay * stock[t-1] + beta * inflow",
    variables=(
        Variable(name="stock", dimension=D.outcome, initial=0.0, description="the compartment"),
        Variable(name="inflow", dimension=D.currency, role="exogenous"),
    ),
    parameters=(
        Param(name="decay", dimension=NONE),
        Param(name="beta", dimension=D.outcome / D.currency),
    ),
    name="one-compartment",
)
print(system.endogenous, system.exogenous, "| max lag:", system.max_lag)
print([p.name for p in system.parameters])
show_math(system.equation("stock").rhs)

In [ ]:
# References parse both ways, and an equation knows what it reads.
print(lag_ref("stock", 2), parse_ref("stock.l2"), time_ref("stock", 3))
print(refs_in(system.equation("stock").rhs))
print(system.equation("stock").refs, "| label:", system.equation("stock").label)
role: Role = system.variable("inflow").role
print("role of inflow:", role)

## What the language refuses

Every refusal names what is wrong and what would fix it. Dimensions are checked across the
equals sign, using the same dimension interpreter the rest of axiom uses — a system that
type-checks here cannot disagree with `forward()` later.

In [ ]:
try:
    parse_system(
        "y = x",
        variables=(
            Variable(name="y", dimension=D.outcome),
            Variable(name="x", dimension=D.currency, role="exogenous"),
        ),
    )
except Exception as e:
    print(type(e).__name__, "->", e)

try:
    parse_ref("y.f1")
except LeadNotSupportedError as e:
    print("lead ->", e)

try:
    parse_equations("y = wobble", variables=(Variable(name="y", dimension=NONE),))
except ParseError as e:
    print("unknown name ->", e)

try:
    DynamicSystem(variables=(Variable(name="y", dimension=NONE),), equations=())
except Exception as e:
    print("missing equation ->", e)

## Blocks: which equations are genuinely simultaneous

The strongly connected components of the contemporaneous dependency graph are exactly the
sets that have to be solved together. A recursive system is already a DAG within a period.
Lagged edges never enter a block — which is precisely why unrolling gives a DAG.

In [ ]:
order: BlockOrder = block_order(system)
print("recursive:", order.recursive, "| blocks:", [(b.variables, b.simultaneous) for b in order.blocks])
first: Block = order.blocks[0]
print("first block size:", first.size, "| solution order:", order.order, "| largest:", order.largest_block)
print(strongly_connected_components(("a", "b", "c"), (("a", "b"), ("b", "a"), ("b", "c"))))

In [ ]:
market = parse_system(
    """
    quantity = a - b * price + c * income
    price    = d + e * quantity + cost
    """,
    variables=(
        Variable(name="quantity", dimension=D.outcome),
        Variable(name="price", dimension=D.currency),
        Variable(name="income", dimension=D.currency, role="exogenous"),
        Variable(name="cost", dimension=D.currency, role="exogenous"),
    ),
    parameters=(
        Param(name="a", dimension=D.outcome),
        Param(name="b", dimension=D.outcome / D.currency),
        Param(name="c", dimension=D.outcome / D.currency),
        Param(name="d", dimension=D.currency),
        Param(name="e", dimension=D.currency / D.outcome),
    ),
    name="market",
)
market_order = block_order(market)
print("recursive:", market_order.recursive)
print("simultaneous:", [b.variables for b in market_order.simultaneous_blocks])
print("cycles through:", market.cycles_through())

## Solving a block

A simultaneous block is solved *symbolically at compile time*. `affine_split` decides,
structurally, whether each right-hand side is affine in the block's unknowns; if it is,
`reduced_form` eliminates them and the result is the econometric reduced form — exact for
every parameter value, and differentiable, so the design math downstream sees the true
derivative.

In [ ]:
from axiom.core import Data, Mul

form: AffineForm | None = affine_split(market.equation("quantity").rhs, ["quantity", "price"])
print("affine in the unknowns:", form is not None, "| coefficients on:", sorted(form.coefficients))

dims = {v.name: v.dimension for v in market.variables}
solved = reduced_form(
    {v: market.equation(v).rhs for v in ("quantity", "price")},
    dims,
)
show_math(solved["quantity"])

In [ ]:
solution: BlockSolution = solve_block(
    {v: market.equation(v).rhs for v in ("quantity", "price")},
    dims,
    simultaneous=True,
)
method: SolveMethod = solution.method
print("method:", method, "| exact:", solution.exact, "| nodes:", solution.node_count)
print("residuals (empty because the solve is exact):", solution.residuals)

### A nonlinear block is approximate, and says so

When a right-hand side is not affine there is no closed-form reduced form. The block is
compiled as a declared number of Gauss-Seidel sweeps, `exact` is false, and `residuals`
carries `rhs(v) - v` per unknown — evaluate it on real data and you get the error you are
actually running. The tree grows *geometrically* in the sweeps, so useful counts are single
digits; past the node budget the answer is `Unsupported`, not a hang.

In [ ]:
saturating = parse_system(
    """
    response = k * load / (1 + load)
    load     = drive + g * response
    """,
    variables=(
        Variable(name="load", dimension=NONE),
        Variable(name="response", dimension=NONE),
        Variable(name="drive", dimension=NONE, role="exogenous"),
    ),
    parameters=(Param(name="k", dimension=NONE), Param(name="g", dimension=NONE)),
    name="saturating-feedback",
)
print("without sweeps:", conditional_form(saturating).reason[:120])

swept = conditional_form(saturating, sweeps=6)
print("with 6 sweeps -> exact:", swept.exact, "| nodes:", swept.node_count)
data, theta = {"drive": np.array([1.0])}, {"k": 2.0, "g": 0.5}
got = float(np.ravel(value(swept.expression("response"), data=data, params=theta))[0])
worst = max(
    abs(float(np.ravel(value(r, data=data, params=theta))[0]))
    for r in swept.approximate_blocks[0].residuals.values()
)
truth = 0.0
for _ in range(400):
    truth = 2.0 * (1 + 0.5 * truth) / (1 + 1 + 0.5 * truth)
print(f"response {got:.6f} vs fixed point {truth:.6f}; largest residual {worst:.2e}")